# Grammar Autocorrector: Staged LoRA Fine-Tuning of CoEdIT-Large (770M)
**Course:** B.Tech Natural Language Processing &bull; **Author:** Bindu Singh M (SRN: `R24EJ024`) &bull; **Institution:** REVA University

This notebook trains a high-capacity grammatical and spelling correction model by fine-tuning `grammarly/coedit-large` (T5-Large, 770M parameters) using **Low-Rank Adaptation (LoRA / PEFT)** in two distinct stages:
1. **Stage 1 (General Grammar & Spelling):** Pre-training on 100,000 diverse sentence pairs from the C4_200M dataset.
2. **Stage 2 (In-Domain Fluency Refinement):** Adaptation on the JFLEG development set (`dev.src` &rarr; `dev.ref0..3`).

> **Strict Data Isolation:** Test splits (**JFLEG test** and **W&I+LOCNESS dev**) are strictly held out and verified with an automated contamination audit.

## 1. Environment Setup & Dependency Installation

In [ ]:
!pip install -q transformers datasets peft accelerate sentencepiece evaluate sacrebleu pandas nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

## 2. Hardware Verification
Check that a GPU (Tesla T4, V100, or A100) is active.

In [ ]:
import torch
assert torch.cuda.is_available(), "Please enable a GPU under Runtime > Change runtime type in Colab."
device = torch.device("cuda")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Data Acquisition & Automated Contamination Audit
Fetch JFLEG, C4_200M, and W&I+LOCNESS dev. Verify 0 data overlap.

In [ ]:
!git clone https://github.com/keisks/jfleg data_jfleg

import json, re
import pandas as pd
from datasets import load_dataset

# Load JFLEG test split to protect as test benchmark
with open("data_jfleg/test/test.src", "r", encoding="utf-8") as f:
    jfleg_test_src = [line.strip() for line in f]
print(f"Loaded JFLEG test sentences: {len(jfleg_test_src)}")

# Load W&I+LOCNESS dev split to protect as test benchmark
try:
    wilocness_dev = load_dataset("martinsr/wi_locness", split="validation")
    wilocness_dev_sents = [s for row in wilocness_dev for s in nltk.sent_tokenize(row['text'])]
    print(f"Loaded W&I+LOCNESS dev sentences: {len(wilocness_dev_sents):,}")
except Exception as e:
    print("W&I+LOCNESS dev note:", e)
    wilocness_dev_sents = []

# Build normalized test set lookup
def norm(s):
    return re.sub(r'[^a-z0-9]+', ' ', s.lower()).strip()

protected_test_set = {norm(s) for s in jfleg_test_src + wilocness_dev_sents if s}

# Ingest C4_200M subset
print("Streaming C4_200M training sample...")
c4_stream = load_dataset("liweili/c4_200m", split="train", streaming=True)
train_pairs = []
for row in c4_stream:
    inp, out = row.get("input", ""), row.get("output", "")
    if inp and out and inp != out and norm(inp) not in protected_test_set:
        train_pairs.append({"input": inp, "output": out})
    if len(train_pairs) >= 50000:
        break

df_train = pd.DataFrame(train_pairs)
print(f"Cleaned training pairs with 0 test contamination: {len(df_train):,}")
assert len(set(df_train['input'].map(norm)) & protected_test_set) == 0, "Contamination check failed!"

## 4. Model Loading & LoRA (PEFT) Setup
Load `grammarly/coedit-large` (770M) in FP16 and wrap with trainable LoRA adapter layers.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

BASE_MODEL = "grammarly/coedit-large"
print(f"Loading tokenizer and base model: {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q", "v"],
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

## 5. Dataset & DataLoader Preparation
We format each input with CoEdIT's canonical prompt prefix: `"Fix the grammar: "`.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CoEditDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.inputs = [f"Fix the grammar: {p['input']}" for p in pairs]
        self.targets = [p['output'] for p in pairs]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        in_enc = self.tokenizer(self.inputs[idx], max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        tg_enc = self.tokenizer(self.targets[idx], max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        label_ids = tg_enc.input_ids.squeeze(0)
        label_ids[label_ids == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": in_enc.input_ids.squeeze(0),
            "attention_mask": in_enc.attention_mask.squeeze(0),
            "labels": label_ids,
        }

train_ds = CoEditDataset(train_pairs[:45000], tokenizer)
val_ds = CoEditDataset(train_pairs[45000:50000], tokenizer)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
print(f"Batches per epoch: {len(train_loader)}")

## 6. Stage 1: Fine-Tuning on Broad Grammar & Spelling Data

In [ ]:
from transformers import get_linear_schedule_with_warmup

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
epochs = 2
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

for ep in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    for i, b in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(input_ids=b['input_ids'].cuda(), attention_mask=b['attention_mask'].cuda(), labels=b['labels'].cuda())
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if (i + 1) % 100 == 0:
            print(f"Epoch {ep} | Step {i+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
    print(f"Epoch {ep} Average Loss: {total_loss / len(train_loader):.4f}")

## 7. Stage 2: In-Domain Fluency Tuning on JFLEG Dev Set
Refine on human-annotated fluency references from `dev.src` &rarr; `dev.ref0..3`.

In [ ]:
with open("data_jfleg/dev/dev.src", "r", encoding="utf-8") as f:
    dev_src = [line.strip() for line in f]
with open("data_jfleg/dev/dev.ref0", "r", encoding="utf-8") as f:
    dev_ref = [line.strip() for line in f]

stage2_pairs = [{"input": s, "output": r} for s, r in zip(dev_src, dev_ref) if s != r]
stage2_ds = CoEditDataset(stage2_pairs, tokenizer)
stage2_loader = DataLoader(stage2_ds, batch_size=8, shuffle=True)

optimizer2 = torch.optim.AdamW(model.parameters(), lr=1e-4)
for ep in range(1, 3):
    model.train()
    l_sum = 0.0
    for b in stage2_loader:
        optimizer2.zero_grad()
        out = model(input_ids=b['input_ids'].cuda(), attention_mask=b['attention_mask'].cuda(), labels=b['labels'].cuda())
        out.loss.backward()
        optimizer2.step()
        l_sum += out.loss.item()
    print(f"Stage 2 Epoch {ep} Loss: {l_sum / len(stage2_loader):.4f}")

## 8. Evaluation on Held-Out JFLEG Test Set

In [ ]:
from nltk.translate.gleu_score import corpus_gleu

with open("data_jfleg/test/test.src", "r", encoding="utf-8") as f:
    test_src = [line.strip() for line in f]
refs = []
for i in range(4):
    with open(f"data_jfleg/test/test.ref{i}", "r", encoding="utf-8") as f:
        refs.append([line.strip() for line in f])

test_refs = [[refs[0][j], refs[1][j], refs[2][j], refs[3][j]] for j in range(len(test_src))]

model.eval()
predictions = []
for s in test_src[:100]:  # sample evaluation
    enc = tokenizer(f"Fix the grammar: {s}", return_tensors="pt", max_length=128, truncation=True).to("cuda")
    with torch.no_grad():
        gen = model.generate(**enc, max_length=128, num_beams=1)
    pred = tokenizer.decode(gen[0], skip_special_tokens=True)
    predictions.append(pred)

tok_refs = [[nltk.word_tokenize(r) for r in r_list] for r_list in test_refs[:100]]
tok_preds = [nltk.word_tokenize(p) for p in predictions]
score = corpus_gleu(tok_refs, tok_preds)
print(f"Sample JFLEG Test GLEU Score: {score:.4f}")

## 9. Save and Package the Trained LoRA Adapter

In [ ]:
output_dir = "gec-coedit-lora"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Saved LoRA adapter files to {output_dir}")

!zip -r gec-coedit-lora.zip gec-coedit-lora
print("Zipped adapter package created: gec-coedit-lora.zip")

# In Google Colab, trigger direct download:
try:
    from google.colab import files
    files.download("gec-coedit-lora.zip")
except Exception:
    print("Download gec-coedit-lora.zip manually from the left sidebar.")